In [0]:
PATH_CUSTOMER_FILES=f"/Volumes/customer_360/raw/source_files/landing_data/customers/"
TABLE_BRONZE_CUSTOMER =f"customer_360.bronze.customers"
TABLE_METRIC ="customer_360.raw.stream_metrics"
PATH_CUSTOMER_CHECKPOINTLOCATION_BRONZE="/Volumes/customer_360/raw/source_files/checkpoints/customers/"

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_BRONZE_CUSTOMER} (
    customer_id STRING NOT NULL,
    first_name STRING,
    last_name STRING,
    email STRING,
    phone STRING,
    city STRING,
    region STRING,
    customer_segment STRING,
    registration_date DATE,
    updated_at TIMESTAMP
)
USING DELTA
""")
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE_METRIC} (
    metric_time TIMESTAMP NOT NULL,
    query_name STRING NOT NULL,
    batch_id BIGINT NOT NULL,
    input_rows BIGINT NOT NULL,
    input_rows_per_second DOUBLE,
    processed_rows_per_second DOUBLE,
    processing_time_ms BIGINT
)
USING DELTA
""")

In [0]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DateType,
    TimestampType,
    LongType,
    DoubleType
)

customer_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("city", StringType(), True),
    StructField("region", StringType(), True),
    StructField("customer_segment", StringType(), True),
    StructField("registration_date", DateType(), True),
    StructField("updated_at", TimestampType(), True)
])

stream_metrics_schema = StructType([
    StructField("metric_time", TimestampType(), False),
    StructField("query_name", StringType(), False),
    StructField("batch_id", LongType(), False),
    StructField("input_rows", LongType(), False),
    StructField("input_rows_per_second", DoubleType(), True),
    StructField("processed_rows_per_second", DoubleType(), True),
    StructField("processing_time_ms", LongType(), True)
])

In [0]:
customer_bronze=(
    spark
    .readStream
    .format("csv")
    .option("header",True)
    .schema(customer_schema)
    .load(PATH_CUSTOMER_FILES)
)

In [0]:
query=(
    customer_bronze
    .writeStream
    .trigger(availableNow=True)
    .format("delta")
    .outputMode("append")
    .option("checkpointlocation",PATH_CUSTOMER_CHECKPOINTLOCATION_BRONZE)
    .toTable(TABLE_BRONZE_CUSTOMER)
)
query.awaitTermination()

In [0]:

import json
from pyspark.sql import Row
from datetime import datetime

metrics = []

for p in query.recentProgress:

    progress = json.loads(p.json)

    source = progress["sources"][0]

    metrics.append(
        Row(
            metric_time=datetime.now(),
            query_name="customer_bronze",
            batch_id=int(progress["batchId"]),
            input_rows=int(source.get("numInputRows", 0)),
            input_rows_per_second=float(source.get("inputRowsPerSecond", 0.0)),
            processed_rows_per_second=float(source.get("processedRowsPerSecond", 0.0)),
            processing_time_ms=int(
                progress.get("durationMs", {}).get("triggerExecution", 0)
            )
        )
    )

if metrics:
    metrics_df = spark.createDataFrame(metrics)

    metrics_df.write \
        .format("delta") \
        .mode("append") \
        .saveAsTable(TABLE_METRIC)

In [0]:
display(
    spark.sql(f"select * from {TABLE_BRONZE_CUSTOMER}")

)